# Simple notebook to check embeddings are correct

Goes through all subfolders in a root folder, reports shape of image and text embeddings as well as how many rows have norm that is not close to 1 (a sanity check)

In [1]:
from pathlib import Path
import numpy as np


def inspect_embedding_norms(root_folder):
    """
    Recursively search root_folder for text_embeddings.npy and img_embeddings.npy.
    For each file found, print:
        folder name | modality | array shape | number of rows with norm outside [1/2, 3/2]
    """

    root_folder = Path(root_folder).expanduser().resolve()

    if not root_folder.exists():
        raise FileNotFoundError(f"Root folder does not exist: {root_folder}")

    filenames_to_modality = {
        "text_embeddings.npy": "text",
        "img_embeddings.npy": "image",

        # Optional typo-tolerant version, in case some files are named this way:
        "text_emeddings.npy": "text",
    }

    found_any = False

    for path in sorted(root_folder.rglob("*.npy")):
        if path.name not in filenames_to_modality:
            continue

        found_any = True
        modality = filenames_to_modality[path.name]
        folder_name = path.parent.name

        try:
            X = np.load(path, allow_pickle=False)
        except Exception as e:
            print(f"{folder_name} | {modality} | LOAD_FAILED | {path} | error={e}")
            continue

        if X.ndim < 2:
            print(f"{folder_name} | {modality} | shape={X.shape} | NOT_2D_ARRAY")
            continue

        row_norms = np.linalg.norm(X, axis=1)
        bad_rows = np.sum((row_norms < 0.5) | (row_norms > 1.5))

        print(
            f"{folder_name} | "
            f"{modality} | "
            f"shape={X.shape} | "
            f"bad_norm_rows={bad_rows}"
        )

    if not found_any:
        print(f"No embedding files found under: {root_folder}")

In [4]:
inspect_embedding_norms("/home/kirilb/orcd/scratch/PRH_data/embedded_words")

BAAI__bge-base-en-v1.5 | text | shape=(50000, 768) | bad_norm_rows=0
BAAI__bge-large-en-v1.5 | text | shape=(50000, 1024) | bad_norm_rows=0
Qwen__Qwen3-1.7B-Base | text | shape=(50000, 2048) | bad_norm_rows=0
Qwen__Qwen3-4B-Base | text | shape=(50000, 2560) | bad_norm_rows=0
codefuse-ai__F2LLM-1.7B | text | shape=(50000, 2048) | bad_norm_rows=0
codefuse-ai__F2LLM-4B | text | shape=(50000, 2560) | bad_norm_rows=0
google__gemma-3-1b-it | text | shape=(50000, 1152) | bad_norm_rows=0
google__gemma-3-4b-it | text | shape=(50000, 2560) | bad_norm_rows=0
meta-llama__Llama-3.2-1B-Instruct | text | shape=(50000, 2048) | bad_norm_rows=0
meta-llama__Llama-3.2-3B-Instruct | text | shape=(50000, 3072) | bad_norm_rows=0
nomic-ai__nomic-embed-text-v1.5 | text | shape=(50000, 768) | bad_norm_rows=0
nomic-ai__nomic-embed-text-v2-moe | text | shape=(50000, 768) | bad_norm_rows=0
